## Download model

In [556]:
from ultralytics import YOLO
import torch

# Downloads automatically from Ultralytics on first run, then cached in ~/.cache/ultralytics
# Creates instance of the YOLO wrapper class, which loads the model
model_name = 'yolov8n.pt'

model = YOLO(model_name)
model.fuse()

# model.model is the actual nn.Module (DetectionModel) inside the YOLO wrapper
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

inner_model = model.model.to(device)
print(f'Inner model type: {type(inner_model).__name__}')

YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
Device: cpu
Inner model type: DetectionModel


## Run calibration

Create model handles to save activations as we go

In [557]:
from pathlib import Path
import urllib
import zipfile
import json as _json

N_CALIBRATION = 64   # paper uses 64 calibration samples

def get_calibration_images(n: int = N_CALIBRATION) -> list:
    """
    Return a list of image file paths for calibration.

    Priority:
      1. Local cocosample/ folder (teammate's original approach)
      2. Download n images from COCO val2017
    """
    local = Path('cocosample')
    if local.exists() and len(list(local.glob('*.jpg'))) > 0:
        imgs = sorted(local.glob('*.jpg'))[:n]
        print(f'Using local cocosample/ — found {len(imgs)} images')
        return [str(p.resolve()) for p in imgs]

    print('cocosample/ not found — downloading COCO val2017 images...')
    download_dir = Path('coco_calib')
    download_dir.mkdir(exist_ok=True)

    base_url = 'http://images.cocodataset.org/val2017/'
    ann_url  = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
    ann_dir  = Path('coco_annotations')

    if not ann_dir.exists():
        print('  Downloading COCO annotations (~250MB, done once)...')
        ann_zip = Path('annotations.zip')
        urllib.request.urlretrieve(ann_url, ann_zip)
        with zipfile.ZipFile(ann_zip, 'r') as z:
            z.extract('annotations/instances_val2017.json', 'coco_annotations')
        ann_zip.unlink()

    with open('coco_annotations/annotations/instances_val2017.json') as f:
        coco = _json.load(f)

    image_files = []
    for img_info in coco['images'][:n]:
        fname = img_info['file_name']
        fpath = download_dir / fname
        if not fpath.exists():
            urllib.request.urlretrieve(base_url + fname, fpath)
        image_files.append(str(fpath.resolve()))

    print(f'  Downloaded {len(image_files)} calibration images to coco_calib/')
    return image_files


calibration = get_calibration_images(N_CALIBRATION)
print(f'Calibration set: {len(calibration)} images')

# Map to save layer activations
activations = {}

def get_activations(name):
    def hook(module, input, output):
        # Handle tensor output or tuple/list output
        if isinstance(output, torch.Tensor):
            out = output
        elif isinstance(output, (tuple, list)) and len(output) > 0:
            out = output[0]
        else:
            return
        if isinstance(out, torch.Tensor):
            activations[name] = out.detach().cpu()
    return hook

handles = []

# FIX: register hooks on model.model (the inner nn.Module)
for name, layer in model.model.named_modules():
    if len(list(layer.children())) == 0:
        handles.append(layer.register_forward_hook(get_activations(name)))

print(f'Registered {len(handles)} hooks on model.model')

cocosample/ not found — downloading COCO val2017 images...
  Downloaded 64 calibration images to coco_calib/
Calibration set: 64 images
Registered 72 hooks on model.model


### Run inference

In [558]:
# Automatically find images — works on any machine
def get_image_paths(folder: str = "cocosample") -> list[str]:
    """
    Returns absolute paths to all jpg/png images in the given folder.
    Uses Path so it works on Windows, Mac, and Linux without changes.
    """
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(
            f"Could not find '{folder}'. "
            f"Current directory is: {os.getcwd()}"
        )
    paths = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png"))
    if not paths:
        raise FileNotFoundError(f"No images found in '{folder}'")
    return [str(p) for p in paths]

calibration_paths = get_image_paths("coco_calib")

# Run inference
for img_path in calibration_paths:
    with torch.no_grad():
        model(img_path)   # pass full path directly — no string concatenation

for h in handles:
    h.remove()

image 1/1 c:\Users\anuhg\Documents\Southampton\_Lessons\Year4\DPDL\Group_Project\DPDLFINALATTEMPT\yoloDGQ\coco_calib\000000006818.jpg: 640x448 1 toilet, 324.8ms
Speed: 3.2ms preprocess, 324.8ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 448)

image 1/1 c:\Users\anuhg\Documents\Southampton\_Lessons\Year4\DPDL\Group_Project\DPDLFINALATTEMPT\yoloDGQ\coco_calib\000000016228.jpg: 448x640 12 persons, 1 bench, 1 horse, 1 umbrella, 298.0ms
Speed: 5.3ms preprocess, 298.0ms inference, 3.1ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 c:\Users\anuhg\Documents\Southampton\_Lessons\Year4\DPDL\Group_Project\DPDLFINALATTEMPT\yoloDGQ\coco_calib\000000017627.jpg: 480x640 3 persons, 10 cars, 319.1ms
Speed: 12.3ms preprocess, 319.1ms inference, 3.3ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 c:\Users\anuhg\Documents\Southampton\_Lessons\Year4\DPDL\Group_Project\DPDLFINALATTEMPT\yoloDGQ\coco_calib\000000025560.jpg: 480x640 1 person, 1 cat, 1 cup, 1 tv, 2825

## Create and save activations outliers and groupings. 

We can save and reuse as outlier positions are stable across inferencing, depending on the strength of the calibration

In [ ]:
from sklearn.cluster import KMeans
import json
from tqdm import tqdm

n_groups = 4

QUANT_PARAMS = {}

for name, layer in tqdm(model.model.named_modules()):
    if len(list(layer.children())) == 0:
        if isinstance(layer, torch.nn.Conv2d):
            if layer.out_channels > 1:
                layer_acts   = activations[name].squeeze(0)
                channels_dim = layer_acts.view(layer_acts.shape[0], -1)

                kmeans = KMeans(n_clusters=n_groups, random_state=0, n_init='auto')
                kmeans.fit(channels_dim)
                groups = kmeans.predict(channels_dim)

                # Build per-group min/max — logic unchanged from original
                quant_groups = {}
                for i in range(len(groups)):
                    g = groups[i]
                    if g in quant_groups:
                        if max(channels_dim[i]) > quant_groups[g]['max']:
                            quant_groups[g]['max'] = torch.max(channels_dim[i]).detach().item()
                        if min(channels_dim[i]) < quant_groups[g]['min']:
                            quant_groups[g]['min'] = torch.min(channels_dim[i]).detach().item()
                    else:
                        quant_groups[g] = {'min': torch.min(channels_dim[i]).detach().item(), 'max': torch.max(channels_dim[i]).detach().item()}
            else:
                print(f"  Skipping {name} — only {channels_dim.shape[0]} channels, less than n_groups={n_groups}")
           
            channel_quant = dict(zip(range(len(channels_dim)), [quant_groups[i] for i in groups]))

            QUANT_PARAMS[name] = channel_quant


with open(f"{model_name.replace('.pt', '')}_{n_groups}_quant_params.json", 'w+') as fp:
    json.dump(QUANT_PARAMS, fp)


127it [03:47,  1.63it/s]

## Create DGQ Layer wrapper

Allows us to quantise activations by channel groups during inference, and weight quantization at initialisation

In [599]:
import numpy as np  

class DGQCNNWrapper(torch.nn.Module):

    def __init__(self, layer, layer_params, weight_bits, act_bits):
        super().__init__()
        self.layer        = layer
        self.layer_params = layer_params
        self.act_bits     = act_bits

        self.out_channels = layer.out_channels
        self.in_channels  = layer.in_channels
        self.bias         = layer.bias
        self.stride       = layer.stride
        self.padding      = layer.padding
        self.dilation     = layer.dilation
        self.groups       = layer.groups

        #Weight Quantisation
        self.weight       = layer.weight
    
        v_w = self.weight.view(-1,1).squeeze(1)


   
        q_s = (torch.max(v_w) - torch.min(v_w)) / torch.tensor(2**weight_bits-1)
        q_z = torch.max(torch.tensor(0), torch.min(torch.tensor(2**weight_bits-1), torch.round(-(torch.min(v_w)/q_s))))

        self.weight = torch.nn.Parameter((torch.round(self.weight/q_s + q_z) - q_z)*q_s)


    def forward(self, x):
        conv = torch.nn.functional.conv2d(x, self.weight, 
                                          bias=self.bias, 
                                          padding=self.padding, 
                                          stride=self.stride, 
                                          dilation=self.dilation, 
                                          groups=self.groups)
        
        v_c  = conv.view(conv.shape[0], -1)

        for i in range(len(v_c)):
            q_params = self.layer_params[str(i)]
            q_s = torch.tensor((q_params['max'] - q_params['min']) / (2**self.act_bits-1))
            q_z = torch.max(torch.tensor(0), torch.min(torch.tensor(2**self.act_bits-1), torch.round(torch.tensor(-q_params['min'])/q_s)))

            q_c = (torch.round(v_c[i]/q_s+q_z)-q_z)*q_s

            v_c[i] = q_c

        conv = v_c.view(conv.shape)

        return conv


### Load saved activation groups

In [587]:
def load_params(filename):

    x = dict()

    with open(filename, 'r') as fp:
        x = json.load(fp)

    return x


### Iterate through layers in the model and wrap them. Only quantising CNN layers.

In [604]:
def WrapLayers(model, params, weight_bits, act_bits, name_prefix=""):
    
    for name, module in model.named_children():

        full_name = f"{name_prefix}.{name}" if name_prefix else name

        if isinstance(module, torch.nn.Conv2d):
            
            if module.out_channels>1:
                
                #Get quant params
                #print(full_name)
                layer_params  = params[full_name]
                wrapped_layer = DGQCNNWrapper(module, layer_params, weight_bits, act_bits)
                #print(type(wrapped_layer))
                
                setattr(model, name, wrapped_layer)

        else:
            WrapLayers(module, params, weight_bits, act_bits, full_name)

model_quant = YOLO(model_name)
model_quant.fuse()

inner_model_quant = model_quant.model.to(device)

params = load_params("yolov8n_2_quant_params.json")
WrapLayers(inner_model_quant, params, 8, 8)

results=model_quant("C:/Users/anuhg/Documents/Southampton/_Lessons/Year4/DPDL/Group_Project/DPDLFINALATTEMPT/coco_calib/000000456496.jpg")

print(results)

for i, r in enumerate(results):
    r.show()

YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

image 1/1 C:\Users\anuhg\Documents\Southampton\_Lessons\Year4\DPDL\Group_Project\DPDLFINALATTEMPT\coco_calib\000000456496.jpg: 448x640 1 person, 4 birds, 555.0ms
Speed: 7.5ms preprocess, 555.0ms inference, 6.2ms postprocess per image at shape (1, 3, 448, 640)
[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'basebal